# pointcloud2ifc Demo

This notebook demonstrates the full **point cloud to IFC/BIM** conversion pipeline provided by `pointcloud2ifc`.

## Overview

The library converts 3D point cloud scans (PLY, PCD, LAS/LAZ) into IFC (Industry Foundation Classes) files — the open standard for BIM (Building Information Modeling). The pipeline consists of three stages:

1. **Load** a point cloud from disk (or create one programmatically)
2. **Segment** the point cloud into building elements using one of the available methods:
   - `ransac` — iterative RANSAC plane fitting with orientation-based classification
   - `dbscan` — DBSCAN spatial clustering with geometric heuristics
   - `ml` — PointNet-based semantic segmentation (requires PyTorch + pretrained weights)
3. **Build** an IFC model where each segment becomes a typed `IfcBuildingElement`

Segments are labelled using the **BIMNet 14-category** semantic taxonomy (wall, floor, ceiling, door, window, column, beam, stair, railing, furniture, slab, curtain wall, roof, other).

In [ ]:
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt

from pointcloud2ifc import BIMNET_CATEGORIES
from pointcloud2ifc.segmentation import segment
from pointcloud2ifc.ifc_builder import build_ifc_model
from pointcloud2ifc.io import write_ifc

print("Imports OK")

## 1. Create a Synthetic Room Point Cloud

We generate a simple room consisting of a floor, a ceiling, and four walls. Each surface is sampled with ~500 points plus noise, giving roughly 3000 points in total. Normals are estimated after construction.

In [ ]:
rng = np.random.default_rng(42)

# Room dimensions (metres)
W, D, H = 5.0, 4.0, 3.0
N_PER_SURFACE = 500
NOISE = 0.01  # metres

def _plane(x_range, y_range, z_range, n=N_PER_SURFACE):
    """Sample n points uniformly on a rectangular region with small noise."""
    pts = np.column_stack([
        rng.uniform(*x_range, size=n),
        rng.uniform(*y_range, size=n),
        rng.uniform(*z_range, size=n),
    ])
    pts += rng.normal(0, NOISE, pts.shape)
    return pts

# Floor  (z = 0)
floor   = _plane((0, W), (0, D), (0.0, 0.0))
# Ceiling (z = H)
ceiling = _plane((0, W), (0, D), (H, H))
# Wall 1: y = 0
wall1   = _plane((0, W), (0.0, 0.0), (0, H))
# Wall 2: y = D
wall2   = _plane((0, W), (D, D), (0, H))
# Wall 3: x = 0
wall3   = _plane((0.0, 0.0), (0, D), (0, H))
# Wall 4: x = W
wall4   = _plane((W, W), (0, D), (0, H))

all_points = np.vstack([floor, ceiling, wall1, wall2, wall3, wall4])
print(f"Total points: {len(all_points)}")

# Build Open3D point cloud and estimate normals
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(all_points)
pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.3, max_nn=30))

print(f"Has normals: {pcd.has_normals()}")

## 2. RANSAC Segmentation

RANSAC iteratively fits planes to the point cloud. Each extracted plane is classified by its normal direction:
- Near-horizontal (|n_z| > 0.8) → **floor** or **ceiling** depending on height
- Near-vertical (|n_z| < 0.3) → **wall**
- Otherwise → **other**

In [ ]:
ransac_segments = segment(pcd, method="ransac")

print(f"RANSAC detected {len(ransac_segments)} segments:\n")
for i, seg in enumerate(ransac_segments):
    print(f"  [{i}] label={seg.label:<10s}  label_id={seg.label_id}  points={len(seg.points)}")

## 3. DBSCAN Segmentation (Comparison)

DBSCAN clusters spatially close points first, then applies the same orientation heuristics per cluster. This is a purely geometric baseline — it does not assume the scene is composed of planes.

In [ ]:
dbscan_segments = segment(pcd, method="dbscan")

print(f"DBSCAN detected {len(dbscan_segments)} segments:\n")
for i, seg in enumerate(dbscan_segments):
    print(f"  [{i}] label={seg.label:<10s}  label_id={seg.label_id}  points={len(seg.points)}")

## 4. Build IFC Model from RANSAC Segments

Each segment is converted into a typed IFC building element with bounding-box geometry:

| Segment label | IFC class |
|---|---|
| wall | `IfcWall` |
| floor | `IfcSlab` |
| ceiling | `IfcCovering` |

The resulting `.ifc` file can be opened in any BIM viewer (BIMvision, FreeCAD, Blender with IFC add-on, etc.).

In [ ]:
from pathlib import Path

ifc_model = build_ifc_model(ransac_segments)

output_path = Path("demo_output.ifc")
write_ifc(ifc_model, output_path)

print(f"IFC file saved to: {output_path.resolve()}")
print(f"IFC schema: {ifc_model.schema}")
print(f"Building elements: {len(ifc_model.by_type('IfcBuildingElement'))}")
for elem in ifc_model.by_type("IfcBuildingElement"):
    print(f"  {elem.is_a():<20s}  {elem.Name}")

## 5. Visualization

3D scatter plot of the RANSAC segmentation result. Each colour represents a different segment label.

In [ ]:
LABEL_COLORS = {
    "floor":   "#2ca02c",
    "ceiling": "#1f77b4",
    "wall":    "#d62728",
    "other":   "#7f7f7f",
}

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

for seg in ransac_segments:
    color = LABEL_COLORS.get(seg.label, "#7f7f7f")
    ax.scatter(
        seg.points[:, 0],
        seg.points[:, 1],
        seg.points[:, 2],
        c=color,
        s=1,
        label=seg.label,
        alpha=0.6,
    )

# De-duplicate legend entries
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), loc="upper left")

ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_zlabel("Z (m)")
ax.set_title("RANSAC Segmentation Result")
plt.tight_layout()
plt.show()

## 6. BIMNet Category Reference

The full BIMNet 14-category taxonomy used by `pointcloud2ifc`. The `ml` segmentation backend predicts one of these classes per point.

In [ ]:
IFC_CLASS_MAP = {
    "wall": "IfcWall",
    "floor": "IfcSlab",
    "ceiling": "IfcCovering",
    "door": "IfcDoor",
    "window": "IfcWindow",
    "column": "IfcColumn",
    "beam": "IfcBeam",
    "stair": "IfcStairFlight",
    "railing": "IfcRailing",
    "furniture": "IfcFurnishingElement",
    "slab": "IfcSlab",
    "curtain_wall": "IfcCurtainWall",
    "roof": "IfcRoof",
    "other": "IfcBuildingElementProxy",
}

print(f"{'ID':>4s}  {'Category':<15s}  {'IFC Class'}")
print("-" * 45)
for cat_id, cat_name in BIMNET_CATEGORIES.items():
    ifc_cls = IFC_CLASS_MAP.get(cat_name, "—")
    print(f"{cat_id:4d}  {cat_name:<15s}  {ifc_cls}")